# Tutorial 04: Youtube and Tiktok API
Author: Maximilian Kreutner

In the tutorial this week we will learn how to use the YouTube Data API to search videos, extract statistics and manage how to handle API limits.

<!-- ## What we will cover:
1. Setting up the API Client and Securing your API Key
2. Understanding API Quotas
3. Searching for videos
4. **Basic Function 2:** Getting specific video statistics
5. **Best Practice:** Handling Pagination -->

You need to install the following modules for this tutorial.

In [ ]:
# Install the necessary dependencies
# !pip install google-api-python-client python-dotenv pandas isodate

## Youtube

### Creating and loading an API key

To access public data (like searching for videos, reading public comments, or getting channel statistics), you need an API Key.

1. Go to the [Google Cloud Console](https://console.cloud.google.com/).

2. Click on the project drop-down at the top left and click New Project. Give it a name and create it.

3. Once the project is created, make sure it is selected.

4. In the left sidebar, go to APIs & Services > Library.

5. Search for YouTube Data API v3 and click Enable.

6. Go to APIs & Services > Credentials.

7. Click Create Credentials at the top and select API Key.

Copy your new API key. Do not share it with anyone!

For many APIs you will get your own API key. 
In projects with bigger teams you have to decide if either everyone uses the same API key or all of you use their own.

If everyone uses their own key, good practice is to load the environment from a `.env` file that for example looks like this:

``
YOUTUBE_API_KEY=AIYOURAPIKEYHERE
``

Then you add the .env file to the .gitignore, so you don't upload it onto the public GitHub server.

In [ ]:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build

# This looks for a .env file and loads the variables inside it
load_dotenv()

API_KEY = os.getenv('YOUTUBE_API_KEY')
api_service_name = "youtube"
api_version = "v3"


if not API_KEY:
    print("Error: API Key not found. Make sure your .env file is set up correctly!")
else:
    youtube = build(api_service_name, api_version, developerKey=API_KEY)
    print("API Key loaded securely and YouTube Client created!")

### YouTube API Quotas

API Usage is usually either limited or costly.

For Youtube ach free API key receives **10,000 Quota Units per day** for free. However, different actions cost different amounts of units:
* Searching for videos (`search().list`): **100 units**
* Reading video/channel details (`videos().list`): **1 unit**
* Reading comments (`comments().list`): **1 unit**

You can find a full list of cost for each request on here: https://developers.google.com/youtube/v3/determine_quota_cost

### Searching for Videos

You can search for videos directly within the API.

Let's see how the online presence on youtube of our university looks like.

We want to search for the top 5 videos that show up when searching for 'University of mannheim'. For this we can create a API request with `youtube.search().list` with `type=video`.

Then we can get a response with `request.execute()`.

Note that searching videos is a lot more expensive than getting information about videos where you already know the URL. So if you know beforehand which videos you want to analyze you can save a lot of API Quota units.

In [ ]:
query = "University of mannheim"
max_results = 5

# TODO Add your code here

We should now have a response that contains information about 5 videos in in JSON format.

Transform that data into a `pd.DataFrame`, where each row contains the `Title`, `Description`, `Video ID` and the `Channel` of the video.

For this we can loop through the response with `response.get('items', [])`

We can also get the thumbnails of each video and display it with `IPython.display`. We could for example analyze Thumbnails this way.

In [ ]:
from IPython.display import Image, display
import pandas as pd

videos = []

# TODO add your code here

df = pd.DataFrame(videos)

### Video Statistics
Search results only give us the `snippet` (titles, descriptions, thumbnails). What if we want to know how many **views** or **likes** a video has?

We have to use a different endpoint: `videos().list()`.

*This has a way cheaper cost: 1 quota unit per list call.*
And we can get a total of 50 videos in each call.
*Every day we can get information for 500.000 videos for free.*

Implement the method `get_video_stats` that takes previous `df` as input and appends `likes`, `duration`, `views`, `favorites`, and `comments` to each row.

In [ ]:
def get_video_stats(df, youtube):
    # TODO add your code here
    pass

df_statistics = get_video_stats(df, youtube)

To parse the correct duration we can use `isodate.parse_duration`.

In [ ]:
import isodate

# Optional TODO here

We now have a full dataframe that contains information about views, likes and comments for our videos.

In [ ]:
df_statistics

### Video Comments

These statistics tell us how much reach a certain video has. We can also infer how popular a video is from the ratio of views/likes.

Another interesting aspect is the sentiment in the comments. Are comments favorable or unvaforable towards the video?

For this we can use the method `youtube.commentThreads().list`.

Implement a method get_video_comments, that takes as input the `df_statistics` and creates a new DataFrame that contains information about the comment (e.g. `comment_id`, `author`, `comment_text`, `like_count`, `published_at`) and for each comment records the `video_ID` the comment is from.

*We can get up to 100 top level comments in one request at a cost of 1 quota.*

If there are more than 100 comments on a video we have to go to the next page via the `nextPageToken`.

In [ ]:
def get_video_comments(df_statistics, youtube):
    # TODO add your code here
    pass

df_comment = get_video_comments(df_statistics, youtube)

We can now look at the comments and see if a certain video has positive or negative comments, and what topics people are talking about.

In future exercises we will look at methods to analyze text and comments at scale.

In [ ]:
df_comment.head(5)

In [ ]:
df_comment.head(5).comment_text.to_list()

## TikTok

Unfortunately Tiktok's official [Research API](https://developers.tiktok.com/products/research-api/) is quite restrictive.
It is unlikely that you will get access to it (however I encourage you to try and tell me if it works out for you).

There are unofficial alternatives that are based on scraping: https://github.com/davidteather/TikTok-Api
The repository contains getting started tutorials and also a deeper dive into how it works.

However, keep in mind that TikTok is aware of such unofficial APIs and might ban you if you try to do this with your own personal account. 